# Predicting Diabetes Risk: A Machine Learning Classification Workflow

**Author:** Sébastien Bodrero
**Programme:** MSc in Artificial Intelligence — Woolf University / Udacity
**Module 3:** Machine Learning Foundations
**Date:** April 2026

---

This notebook implements a complete supervised machine learning workflow applied to the **Pima Indians Diabetes Dataset** (768 rows × 9 columns). The task is **binary classification**: predict whether a patient is likely to have diabetes (Outcome = 1) based on eight diagnostic measurements.

The workflow follows six structured sections:

1. **Setup** — library imports and version reporting
2. **Data Ingestion** — auto-download and initial inspection
3. **Data Preparation & Preprocessing** — handling zero-encoded missing values, scaling, train/test split
4. **Model Selection & Training** — Logistic Regression (baseline) and Random Forest (primary model)
5. **Evaluation** — metrics, confusion matrix, ROC curve, feature importance
6. **Notebook Summary** — findings, challenges, and limitations

## 1. Setup

In [1]:
import urllib.request
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
import sklearn

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print(f'NumPy      {np.__version__}')
print(f'Pandas     {pd.__version__}')
print(f'Matplotlib {matplotlib.__version__}')
print(f'Seaborn    {sns.__version__}')
print(f'scikit-learn {sklearn.__version__}')

NumPy      2.4.4
Pandas     3.0.2
Matplotlib 3.10.8
Seaborn    0.13.2
scikit-learn 1.8.0


## 2. Data Ingestion

The **Pima Indians Diabetes Dataset** was originally compiled by the National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK) and made publicly available through the UCI Machine Learning Repository and Kaggle. It contains diagnostic measurements for 768 female patients of Pima Indian heritage aged 21 and above.

| Column | Type | Description |
|--------|------|-------------|
| `Pregnancies` | int | Number of pregnancies |
| `Glucose` | int | Plasma glucose concentration (2-hour oral glucose tolerance test) |
| `BloodPressure` | int | Diastolic blood pressure (mm Hg) |
| `SkinThickness` | int | Triceps skin fold thickness (mm) |
| `Insulin` | int | 2-hour serum insulin (μU/ml) |
| `BMI` | float | Body mass index (kg/m²) |
| `DiabetesPedigreeFunction` | float | Genetic risk score based on family history |
| `Age` | int | Age in years |
| `Outcome` | int | Target: 1 = diabetes, 0 = no diabetes |

The dataset is downloaded automatically if not already present locally.

In [2]:
DATA_URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
DATA_FILE = 'diabetes.csv'

COLUMN_NAMES = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

if not os.path.exists(DATA_FILE):
    print(f'Downloading {DATA_FILE} ...')
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print('Download complete.')
else:
    print(f'{DATA_FILE} already present — skipping download.')

df = pd.read_csv(DATA_FILE, header=None, names=COLUMN_NAMES)
print(f'\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

diabetes.csv already present — skipping download.

Dataset shape: 768 rows × 9 columns


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


**Initial observations:**

- All columns are numeric with no `NaN` values in the raw load — however, the minimum values for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` are **0**, which is physiologically impossible. These zeros represent missing values encoded as zero at the time of data collection, a well-documented issue with this dataset.
- The target class (`Outcome`) is imbalanced: roughly 65% negative (no diabetes) vs 35% positive (diabetes). This imbalance will inform metric selection in Section 5.

In [5]:
print('=== Target class distribution ===')
counts = df['Outcome'].value_counts()
print(counts)
print(f'\nPositive rate (diabetes): {counts[1] / len(df):.1%}')

=== Target class distribution ===
Outcome
0    500
1    268
Name: count, dtype: int64

Positive rate (diabetes): 34.9%


## 3. Data Preparation & Preprocessing

### 3.1 Handling Zero-Encoded Missing Values

Five features contain physiologically impossible zero values that represent missing data: `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. `Pregnancies` (0 = never pregnant) and `Age` (all patients ≥ 21) are legitimate at their minimum values.

**Strategy:** Replace zeros in the five affected columns with `NaN`, then impute with the **column median** (preferred over mean because the distributions are right-skewed, as `Insulin` and `SkinThickness` demonstrate). Median imputation is a standard, robust choice for skewed health data with modest missingness rates (less than 50% per column).

In [6]:
ZERO_IMPUTE_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

df_clean = df.copy()

# Replace physiologically impossible zeros with NaN
df_clean[ZERO_IMPUTE_COLS] = df_clean[ZERO_IMPUTE_COLS].replace(0, np.nan)

print('=== Missing values after zero-replacement ===')
missing = df_clean.isnull().sum()
print(missing[missing > 0])
print(f'\nTotal cells affected: {missing.sum()} / {df_clean.shape[0] * len(ZERO_IMPUTE_COLS)} ({missing.sum() / (df_clean.shape[0] * len(ZERO_IMPUTE_COLS)):.1%})')

=== Missing values after zero-replacement ===
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

Total cells affected: 652 / 3840 (17.0%)


In [7]:
# Impute with column median
for col in ZERO_IMPUTE_COLS:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f'  {col}: imputed {missing[col]} zeros with median = {median_val:.2f}')

print(f'\nRemaining NaNs: {df_clean.isnull().sum().sum()}')

  Glucose: imputed 5 zeros with median = 117.00
  BloodPressure: imputed 35 zeros with median = 72.00
  SkinThickness: imputed 227 zeros with median = 29.00
  Insulin: imputed 374 zeros with median = 125.00
  BMI: imputed 11 zeros with median = 32.30

Remaining NaNs: 0
